<h1 style="text-align:center;"><b>Laboratorio 7</b></h1>
<h3 style="text-align:center;">Marcos Díaz (221102), Daniel Machic (22118), Maria Jose Ramírez (221051)</h3>

**GitHub**: https://github.com/mac2218/IA-Lab07.git


## 1. Clases de Tic Tac Toe

In [3]:
import math
import time
import random
from copy import deepcopy


class TicTacToeEngine:

    # Inicializa tablero 3x3 o 4x4
    def __init__(self, size=3):
        if size not in [3, 4]:
            raise ValueError("size debe ser 3 o 4")

        self.size = size
        self.board = [[" " for _ in range(size)] for _ in range(size)]
        self.nodes = 0

    # Reinicia contador de nodos
    def reset_nodes(self):
        self.nodes = 0

    # Copia del estado actual
    def clone(self):
        new_game = TicTacToeEngine(self.size)
        new_game.board = deepcopy(self.board)
        return new_game

    # Celda disponible
    def is_empty(self, row, col):
        return self.board[row][col] == " "

    # Movimientos disponibles
    def get_moves(self):
        return [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.is_empty(r, c)
        ]

    # Colocar ficha
    def make_move(self, row, col, player):
        self.board[row][col] = player

    # Deshacer movimiento
    def undo_move(self, row, col):
        self.board[row][col] = " "

    # Tablero lleno
    def is_full(self):
        return len(self.get_moves()) == 0

    # Verifica ganador
    def is_winner(self, player):
        n = self.size

        # Filas
        for r in range(n):
            if all(self.board[r][c] == player for c in range(n)):
                return True

        # Columnas
        for c in range(n):
            if all(self.board[r][c] == player for r in range(n)):
                return True

        # Diagonal principal
        if all(self.board[i][i] == player for i in range(n)):
            return True

        # Diagonal secundaria
        if all(self.board[i][n - 1 - i] == player for i in range(n)):
            return True

        return False

    # Estado terminal
    def terminal(self):
        return (
            self.is_winner("X")
            or self.is_winner("O")
            or self.is_full()
        )

    # Función heurística
    def evaluate(self):
        if self.is_winner("X"):
            return 1000

        if self.is_winner("O"):
            return -1000

        score = 0
        n = self.size
        lines = []

        # Filas
        for r in range(n):
            lines.append([self.board[r][c] for c in range(n)])

        # Columnas
        for c in range(n):
            lines.append([self.board[r][c] for r in range(n)])

        # Diagonales
        lines.append([self.board[i][i] for i in range(n)])
        lines.append([self.board[i][n - 1 - i] for i in range(n)])

        for line in lines:
            x = line.count("X")
            o = line.count("O")

            if x > 0 and o == 0:
                score += 10 ** x

            elif o > 0 and x == 0:
                score -= 10 ** o

        return score

    # Minimax puro
    def minimax_pure(self, maximizing=True):
        self.nodes += 1

        if self.terminal():
            return self.evaluate(), None

        moves = self.get_moves()

        if maximizing:
            best_score = -float("inf")
            best_move = None

            for move in moves:
                r, c = move
                self.make_move(r, c, "X")

                score, _ = self.minimax_pure(False)

                self.undo_move(r, c)

                if score > best_score:
                    best_score = score
                    best_move = move

            return best_score, best_move

        else:
            best_score = float("inf")
            best_move = None

            for move in moves:
                r, c = move
                self.make_move(r, c, "O")

                score, _ = self.minimax_pure(True)

                self.undo_move(r, c)

                if score < best_score:
                    best_score = score
                    best_move = move

            return best_score, best_move

    # Minimax limitado
    def minimax_limit(self, depth, maximizing=True):
        self.nodes += 1

        if depth == 0 or self.terminal():
            return self.evaluate(), None

        moves = self.get_moves()

        if maximizing:
            best_score = -float("inf")
            best_move = None

            for move in moves:
                r, c = move
                self.make_move(r, c, "X")

                score, _ = self.minimax_limit(depth - 1, False)

                self.undo_move(r, c)

                if score > best_score:
                    best_score = score
                    best_move = move

            return best_score, best_move

        else:
            best_score = float("inf")
            best_move = None

            for move in moves:
                r, c = move
                self.make_move(r, c, "O")

                score, _ = self.minimax_limit(depth - 1, True)

                self.undo_move(r, c)

                if score < best_score:
                    best_score = score
                    best_move = move

            return best_score, best_move

    # Alpha-Beta pruning
    def alpha_beta(self, depth, alpha, beta, maximizing=True):
        self.nodes += 1

        if depth == 0 or self.terminal():
            return self.evaluate(), None

        moves = self.get_moves()

        if maximizing:
            best_score = -float("inf")
            best_move = None

            for move in moves:
                r, c = move
                self.make_move(r, c, "X")

                score, _ = self.alpha_beta(
                    depth - 1, alpha, beta, False
                )

                self.undo_move(r, c)

                if score > best_score:
                    best_score = score
                    best_move = move

                alpha = max(alpha, best_score)

                if beta <= alpha:
                    break

            return best_score, best_move

        else:
            best_score = float("inf")
            best_move = None

            for move in moves:
                r, c = move
                self.make_move(r, c, "O")

                score, _ = self.alpha_beta(
                    depth - 1, alpha, beta, True
                )

                self.undo_move(r, c)

                if score < best_score:
                    best_score = score
                    best_move = move

                beta = min(beta, best_score)

                if beta <= alpha:
                    break

            return best_score, best_move

    # Simulación aleatoria
    def random_playout(self, turn):
        sim = self.clone()

        while not sim.terminal():
            move = random.choice(sim.get_moves())
            sim.make_move(move[0], move[1], turn)
            turn = "O" if turn == "X" else "X"

        if sim.is_winner("X"):
            return 1
        if sim.is_winner("O"):
            return -1
        return 0

    # Monte Carlo Tree Search
    def mcts(self, iterations=500, C=math.sqrt(2), player="X"):
        self.reset_nodes()

        moves = self.get_moves()

        if len(moves) == 1:
            return moves[0]

        stats = {
            move: {"wins": 0, "visits": 0}
            for move in moves
        }

        for _ in range(iterations):

            # UCT
            parent_visits = (
                sum(stats[m]["visits"] for m in moves) + 1
            )

            best_move = None
            best_score = -float("inf")

            for move in moves:
                visits = stats[move]["visits"]

                if visits == 0:
                    score = float("inf")
                else:
                    mean_win_rate = stats[move]["wins"] / visits
                    score = mean_win_rate + C * math.sqrt(
                        math.log(parent_visits) / visits
                    )

                if score > best_score:
                    best_score = score
                    best_move = move

            sim = self.clone()
            r, c = best_move
            sim.make_move(r, c, player)

            next_turn = "O" if player == "X" else "X"
            result = sim.random_playout(next_turn)

            if player == "O":
                result *= -1

            stats[best_move]["wins"] += result
            stats[best_move]["visits"] += 1

            self.nodes += 1

        return max(
            moves,
            key=lambda m: stats[m]["visits"]
        )


class GameLoop:

    # Configuración inicial
    def __init__(
        self,
        size=3,
        mode="H-H",
        starting_player="H",
        ia_configs=None
    ):
        self.game = TicTacToeEngine(size)
        self.size = size
        self.mode = mode
        self.starting_player = starting_player

        if ia_configs is None:
            ia_configs = {
                "IA1": {
                    "algorithm": "alpha_beta",
                    "depth": 4,
                    "N": 500,
                    "C": math.sqrt(2)
                },
                "IA2": {
                    "algorithm": "mcts",
                    "depth": 4,
                    "N": 500,
                    "C": math.sqrt(2)
                }
            }

        self.ia_configs = ia_configs

    # Mostrar tablero
    def print_board(self):
        n = self.size

        for r in range(n):
            print(" | ".join(self.game.board[r]))
            if r < n - 1:
                print("-" * (4 * n - 3))
        print()

    # Movimiento humano
    def human_move(self):
        while True:
            try:
                r = int(input("Fila: "))
                c = int(input("Columna: "))

                if (
                    0 <= r < self.size
                    and 0 <= c < self.size
                    and self.game.is_empty(r, c)
                ):
                    return (r, c)
            except:
                pass

            print("Movimiento inválido")

    # Movimiento IA
    def ai_move(self, player, config):
        data = self.ia_configs[config]
        algo = data["algorithm"]

        self.game.reset_nodes()
        start = time.time()

        if algo == "minimax":

            if self.size == 3:
                _, move = self.game.minimax_pure(
                    maximizing=(player == "X")
                )
            else:
                _, move = self.game.minimax_limit(
                    data["depth"],
                    maximizing=(player == "X")
                )

        elif algo == "alpha_beta":

            _, move = self.game.alpha_beta(
                data["depth"],
                -float("inf"),
                float("inf"),
                maximizing=(player == "X")
            )

        elif algo == "mcts":

            move = self.game.mcts(
                iterations=data["N"],
                C=data["C"],
                player=player
            )

        else:
            raise ValueError("Algoritmo inválido")

        elapsed = time.time() - start

        print("Nodos explorados:", self.game.nodes)
        print("Tiempo:", round(elapsed, 4), "seg")

        return move

    # Ejecución del juego
    def play(self):

        # H-H
        if self.mode == "H-H":
            current = "X"

        # H-IA
        elif self.mode == "H-IA":
            current = "X"

        # IA-IA
        elif self.mode == "IA-IA":
            current = "X" if self.starting_player == "IA" else "O"

        else:
            raise ValueError("Modo inválido")

        while not self.game.terminal():

            self.print_board()

            # Humano vs Humano
            if self.mode == "H-H":
                move = self.human_move()

            # Humano vs IA
            elif self.mode == "H-IA":

                if (
                    self.starting_player == "H" and current == "X"
                ) or (
                    self.starting_player == "IA" and current == "O"
                ):
                    move = self.human_move()
                else:
                    move = self.ai_move(current, "IA1")

            # IA vs IA
            else:
                if current == "X":
                    move = self.ai_move("X", "IA1")
                else:
                    move = self.ai_move("O", "IA2")

            self.game.make_move(move[0], move[1], current)
            current = "O" if current == "X" else "X"

        self.print_board()

        if self.game.is_winner("X"):
            print("Ganó X")
        elif self.game.is_winner("O"):
            print("Ganó O")
        else:
            print("Empate")

## 2. Explosión Combinatoria

## 3. Duelo de Algoritmos

## 4. Discusion "Para pensar"